# Pharma Doc QA — Exploration Notebook
Experiments for: chunking strategies, embedding quality, retrieval tuning.

**Sections:**
1. Setup & Imports
2. PDF Extraction Inspection
3. Chunking Experiments
4. Embedding & FAISS Index Build
5. Retrieval Testing
6. Threshold Sensitivity Analysis
7. format_context() Output Preview
8. Ground-Truth QA Validation

## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv

load_dotenv('../.env')

from config import (
    CHUNK_SIZE, CHUNK_OVERLAP, TOP_K,
    CONFIDENCE_THRESHOLD, DATA_RAW_DIR,
    QA_PAIRS_DIR, EMBEDDING_MODEL
)
from ingestion import extract_text_from_pdf, load_all_pdfs, chunk_pages, build_faiss_index, save_index, load_index
from retriever import PharmaRetriever

print(f'Config: CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP}, TOP_K={TOP_K}, THRESHOLD={CONFIDENCE_THRESHOLD}')

## 2. PDF Extraction Inspection

In [ ]:
# List available PDFs
pdf_files = list(Path(DATA_RAW_DIR).glob('*.pdf'))
print(f'PDFs found: {len(pdf_files)}')
for p in pdf_files:
    print(f'  {p.name}')

In [ ]:
# Extract pages from first PDF — inspect structure
if pdf_files:
    sample_pages = extract_text_from_pdf(str(pdf_files[0]))
    print(f'Pages extracted: {len(sample_pages)}')
    print(f'\n--- Page 1 preview (first 500 chars) ---')
    print(sample_pages[0]['text'][:500])

In [ ]:
# Page length distribution
all_pages = load_all_pdfs(DATA_RAW_DIR)
page_lengths = [len(p['text']) for p in all_pages]

df_pages = pd.DataFrame({'page_length': page_lengths})
print(df_pages.describe())

plt.figure(figsize=(8, 4))
plt.hist(page_lengths, bins=30, color='steelblue', edgecolor='white')
plt.axvline(CHUNK_SIZE, color='red', linestyle='--', label=f'CHUNK_SIZE={CHUNK_SIZE}')
plt.xlabel('Characters per page')
plt.ylabel('Count')
plt.title('Page Length Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Chunking Experiments

In [ ]:
# Default config chunking
docs = chunk_pages(all_pages)
chunk_lengths = [len(d.page_content) for d in docs]

print(f'Total chunks: {len(docs)}')
print(f'Avg chunk length: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars')
print(f'\nSample chunk [0]:')
print(docs[0].page_content[:300])
print(f'\nMetadata: {docs[0].metadata}')

In [ ]:
# Compare chunk sizes: 400 / 800 / 1200
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

def chunk_with_size(pages, size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap,
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    out = []
    for page in pages:
        for i, c in enumerate(splitter.split_text(page['text'])):
            out.append(Document(page_content=c, metadata=page))
    return out

experiments = {}
for size in [400, 800, 1200]:
    chunks = chunk_with_size(all_pages, size, size // 8)
    lengths = [len(c.page_content) for c in chunks]
    experiments[size] = {'count': len(chunks), 'avg_len': sum(lengths)/len(lengths), 'chunks': chunks}

for size, stats in experiments.items():
    print(f'chunk_size={size:4d} | chunks={stats["count"]:4d} | avg_len={stats["avg_len"]:.0f}')

In [ ]:
# Chunk length distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors = ['steelblue', 'seagreen', 'darkorange']

for ax, (size, stats), color in zip(axes, experiments.items(), colors):
    lengths = [len(c.page_content) for c in stats['chunks']]
    ax.hist(lengths, bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'chunk_size={size}\nn={stats["count"]}')
    ax.set_xlabel('Chars')
    ax.axvline(size, color='red', linestyle='--', linewidth=1)

axes[0].set_ylabel('Count')
plt.suptitle('Chunk Length Distribution by chunk_size', y=1.02)
plt.tight_layout()
plt.show()

## 4. Embedding & FAISS Index Build

In [ ]:
# Build or load index
INDEX_PATH = '../vector_store/pharma_index'

if Path(INDEX_PATH).exists():
    print('Index exists — loading from disk.')
    vector_store = load_index()
else:
    print('Building index — this calls OpenAI Embeddings API.')
    vector_store = build_faiss_index(docs)
    save_index(vector_store)

print('Vector store ready.')

In [ ]:
# Sanity check: raw FAISS similarity search
test_query = 'What are ICH Q1A stability testing requirements?'
raw_results = vector_store.similarity_search_with_relevance_scores(test_query, k=3)

for i, (doc, score) in enumerate(raw_results, 1):
    print(f'\n[{i}] score={score:.4f} | {doc.metadata.get("source")} p.{doc.metadata.get("page_num")}')
    print(doc.page_content[:250])

## 5. Retrieval Testing

In [ ]:
retriever = PharmaRetriever(vector_store=vector_store)

# Test queries covering different regulatory topics
test_queries = [
    'What are ICH Q1A stability testing requirements?',
    'FDA guidelines for clinical trial phases',
    'GMP manufacturing controls for drug substances',
    'Bioavailability and bioequivalence study design',
    'Adverse event reporting requirements'
]

for query in test_queries:
    chunks = retriever.retrieve(query, top_k=TOP_K)
    above = sum(c.above_threshold for c in chunks)
    top_score = chunks[0].score if chunks else 0
    print(f'Q: {query[:60]:60s} | top_score={top_score:.4f} | above_thresh={above}/{len(chunks)}')

In [ ]:
# Deep dive: single query full output
query = test_queries[0]
chunks = retriever.retrieve(query)

print(f'Query: {query}')
print(f'Chunks returned: {len(chunks)}\n')

for i, c in enumerate(chunks, 1):
    flag = '✓' if c.above_threshold else '✗'
    print(f'[{flag}] Chunk {i} | {c.source} p.{c.page_num} | score={c.score}')
    print(f'  {c.content[:200]}\n')

In [ ]:
# retrieve_above_threshold() vs retrieve() comparison
query = test_queries[0]
all_chunks = retriever.retrieve(query)
thresh_chunks = retriever.retrieve_above_threshold(query)

print(f'retrieve()                 → {len(all_chunks)} chunks')
print(f'retrieve_above_threshold() → {len(thresh_chunks)} chunks')
print(f'Threshold: {CONFIDENCE_THRESHOLD}')

## 6. Threshold Sensitivity Analysis

In [ ]:
# How many chunks pass at different thresholds?
import numpy as np

thresholds = np.arange(0.50, 0.96, 0.05)
results = []

for query in test_queries:
    chunks = retriever.retrieve(query, top_k=TOP_K)
    scores = [c.score for c in chunks]
    for t in thresholds:
        above = sum(s >= t for s in scores)
        results.append({'query': query[:40], 'threshold': round(t, 2), 'chunks_above': above})

df_thresh = pd.DataFrame(results)

plt.figure(figsize=(10, 5))
for query in df_thresh['query'].unique():
    sub = df_thresh[df_thresh['query'] == query]
    plt.plot(sub['threshold'], sub['chunks_above'], marker='o', label=query)

plt.axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--', label=f'Current threshold={CONFIDENCE_THRESHOLD}')
plt.xlabel('Confidence Threshold')
plt.ylabel('Chunks Passing')
plt.title('Threshold Sensitivity: Chunks Passing vs Threshold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Score heatmap across queries x top_k slots
score_matrix = []
for query in test_queries:
    chunks = retriever.retrieve(query, top_k=TOP_K)
    scores = [c.score for c in chunks]
    # pad if fewer than TOP_K returned
    scores += [0.0] * (TOP_K - len(scores))
    score_matrix.append(scores)

df_heat = pd.DataFrame(
    score_matrix,
    index=[q[:35] for q in test_queries],
    columns=[f'Rank {i+1}' for i in range(TOP_K)]
)

plt.figure(figsize=(10, 4))
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='YlGn',
            vmin=0, vmax=1, linewidths=0.5)
plt.title('Retrieval Score Heatmap (query × rank)')
plt.tight_layout()
plt.show()

## 7. format_context() Output Preview

In [ ]:
# What the LLM sees as context
query = test_queries[0]
chunks = retriever.retrieve_above_threshold(query)
context = retriever.format_context(chunks)

print(f'Query: {query}')
print(f'Chunks in context: {len(chunks)}')
print(f'Context length: {len(context)} chars')
print('\n' + '='*60)
print(context)

In [ ]:
# Context length across queries
context_lengths = []
for query in test_queries:
    chunks = retriever.retrieve_above_threshold(query)
    ctx = retriever.format_context(chunks)
    context_lengths.append({'query': query[:40], 'chunks': len(chunks), 'context_chars': len(ctx)})

df_ctx = pd.DataFrame(context_lengths)
print(df_ctx.to_string(index=False))

## 8. Ground-Truth QA Validation

In [ ]:
# Load ground-truth QA pairs
qa_files = list(Path(QA_PAIRS_DIR).glob('*.json'))
print(f'QA pair files found: {len(qa_files)}')

qa_pairs = []
for f in qa_files:
    with open(f) as fh:
        data = json.load(fh)
        if isinstance(data, list):
            qa_pairs.extend(data)
        else:
            qa_pairs.append(data)

print(f'Total QA pairs: {len(qa_pairs)}')
if qa_pairs:
    print(f'Sample: {qa_pairs[0]}')

In [ ]:
# Check if ground-truth answer appears in retrieved context
# Expected QA pair format: {"question": ..., "answer": ..., "source": ...}

def keyword_hit(answer: str, context: str, min_keywords: int = 3) -> bool:
    """Rough check: does context contain key terms from the answer?"""
    keywords = [w.lower() for w in answer.split() if len(w) > 4]
    hits = sum(1 for kw in keywords if kw in context.lower())
    return hits >= min(min_keywords, len(keywords))

validation_results = []
for pair in qa_pairs:
    question = pair.get('question', '')
    answer   = pair.get('answer', '')
    chunks   = retriever.retrieve_above_threshold(question)
    context  = retriever.format_context(chunks)
    top_score = chunks[0].score if chunks else 0.0
    hit = keyword_hit(answer, context)
    validation_results.append({
        'question': question[:60],
        'top_score': top_score,
        'chunks_returned': len(chunks),
        'keyword_hit': hit
    })

df_val = pd.DataFrame(validation_results)
print(df_val)

hit_rate = df_val['keyword_hit'].mean() * 100
print(f'\nKeyword Hit Rate: {hit_rate:.1f}%')

In [ ]:
# Summary bar chart
if not df_val.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Hit rate pie
    hits = df_val['keyword_hit'].sum()
    axes[0].pie(
        [hits, len(df_val) - hits],
        labels=['Hit', 'Miss'],
        colors=['seagreen', 'tomato'],
        autopct='%1.0f%%',
        startangle=90
    )
    axes[0].set_title(f'Keyword Hit Rate (n={len(df_val)})')

    # Top score distribution
    axes[1].hist(df_val['top_score'], bins=10, color='steelblue', edgecolor='white')
    axes[1].axvline(CONFIDENCE_THRESHOLD, color='red', linestyle='--',
                    label=f'Threshold={CONFIDENCE_THRESHOLD}')
    axes[1].set_xlabel('Top Chunk Score')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Top Retrieval Score Distribution')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Failures: questions where retrieval missed
if not df_val.empty:
    misses = df_val[df_val['keyword_hit'] == False]
    print(f'Misses: {len(misses)}')
    print(misses[['question', 'top_score', 'chunks_returned']].to_string(index=False))